# PRÁCTICA 7: SPIDER NIVEL 2

In [30]:
# Importamos las librerías necesarias
import csv
import requests
from bs4 import BeautifulSoup
import csv
import requests
import time
import os

In [32]:
# Definimos el prefijo para construir URLs completas a partir de enlaces relativos de Wikipedia.
BASE_URL = "https://en.wikipedia.org"

# Abrimos el archivo "countries.csv" en modo lectura para cargar los datos de los países y sus URLs.
with open("countries.csv", "r", encoding="utf-8") as csvfile:
    # Creamos un lector de archivos CSV con el delimitador de punto y coma.
    reader = csv.reader(csvfile, delimiter=';')
    # Convertimos todo el contenido del archivo en una lista de filas y lo almacenamos en "country_matrix".
    country_matrix = list(reader)

# Seleccionamos la primera fila de "country_matrix", que contiene los nombres de las columnas.
header = country_matrix[0]  # ["CountryName", "CountryUrl"]

# Obtenemos el resto de las filas (es decir, los datos de los países y sus URLs).
data_rows = country_matrix[1:]

# Creamos un conjunto para rastrear las universidades ya guardadas y evitar duplicados.
saved_universities = set()

# Creamos (o sobrescribimos si ya existe) un archivo CSV llamado "universities.csv" en modo escritura.
with open("universities.csv", "w", newline="", encoding="utf-8") as csvfile:
    # Creamos un objeto escritor de CSV con un delimitador de punto y coma.
    writer = csv.writer(csvfile, delimiter=';')
    # Escribimos la cabecera del archivo, añadiendo columnas para las universidades y sus URLs.
    writer.writerow(["CountryName", "CountryUrl", "UniversityName", "UniversityUrl"])

    # Recorremos las filas de datos obtenidas de "countries.csv".
    for row in data_rows:
        # Extraemos el nombre del país y la URL del país de cada fila.
        country_name, country_url = row

        # Comprobamos si el país actual es Alemania (Germany), ignorando diferencias de mayúsculas.
        if country_name.lower() == "germany":
            # Construimos la URL completa (ya está completa en este caso, pero mantenemos el proceso por consistencia).
            full_url = country_url

            # Realizamos una solicitud para descargar el contenido HTML de la página del país.
            response = requests.get(full_url)
            if response.status_code == 200:  # Verificamos que la solicitud fue exitosa.
                # Convertimos el contenido HTML en un objeto que podemos analizar.
                soup = BeautifulSoup(response.content, "html.parser")

                # Si nos fijamos en la página web, hay una sección, titulada "Universities alphabetically" que contiene una lista 
                # de todas las universidades alemanas ordenadas alfabéticamente, por subescciones. Por ello, definimos una lista donde 
                # guardamos el nombre de estas subsecciones que nos permitirá buscar las mencionadas subsecciones alfabéticas de 
                # de las universidades (A–D, E–H, I–N, O–Z).
                subsections = ["A–D", "E–H", "I–N", "O–Z"]
                for subsection in subsections:
                    # Usamos el identificador de cada subsección para localizarla en el HTML.
                    section = soup.find('h3', id=subsection)

                    # Si encontramos esta subsección, buscamos la lista de universidades asociadas.
                    if section:
                        # Localizamos el elemento <ul> que sigue inmediatamente a esta subsección. Este elemento <ul> se utiliza comúnmente 
                        # para agrupar una colección de elementos relacionados, como un conjunto de enlaces o puntos de una lista. En nuestro
                        # caso, un ejemplo sería:
                        # <ul>
                            #<li><a href="/wiki/Technical_University_of_Ilmenau" class="mw-redirect" title="Technical University of Ilmenau">Technical University of Ilmenau</a></li>
                            #<li><a href="/wiki/Jacobs_University_Bremen" class="mw-redirect" title="Jacobs University Bremen">Jacobs University Bremen</a></li>
                            # ...
                        university_list = section.find_next('ul')

                        # Si existe una lista de universidades, procedemos a extraer los enlaces.
                        if university_list:
                            # Obtenemos todos los elementos <a> dentro de la lista que contienen enlaces. (véase el ejemplo anterior)
                            university_links = university_list.find_all('a', href=True)

                            # Recorremos cada enlace encontrado.
                            for link in university_links:
                                # Extraemos el nombre de la universidad del texto visible del enlace. Por ejemplo, en:
                                # <li><a href="/wiki/University_of_Berlin">University of Berlin</a></li>
                                # el texto visible del enlace sería University of Berlin, que es lo que obtenemos con link.text. 
                                # Además, strip() elimina cualquier espacio en blanco extra al inicio o final del texto. 
                                university_name = link.text.strip()
                                
                                # Construimos la URL completa de la universidad, usando el prefijo BASE_URL.
                                university_url = f"{BASE_URL}{link['href']}"

                                # Comprobamos si la universidad ya ha sido guardada anteriormente.
                                # Esto es necesario porque hay universidades que están en dos subsecciones alfabéticas diferentes.
                                # Por ejemplo, la Jacobs University Bremen aparece tanto en la subsección A–D como en la subsección I–N.
                                if university_name not in saved_universities:
                                    # Añadimos la universidad al conjunto para evitar duplicados.
                                    saved_universities.add(university_name)

                                    # Escribimos los datos de la universidad en el archivo "universities.csv".
                                    writer.writerow([country_name, full_url, university_name, university_url])

# Mostramos un mensaje para indicar que el archivo "universities.csv" se ha creado con éxito.
print("Archivo universities.csv creado con éxito.")

Archivo universities.csv creado con éxito.
